# WLAN-Indoor Positionierung
Dieses Notebook kann universell verwendet werden.

## 1. Server starten
Die Daten werden zentral auf einem Server in eine SQLite Datenbank gespeichert. Zum starten des Server muss die Datei ´´´server/app.py´´´ mit Folgendem Befehl ausgeführt werden:
```bash
python3 server/app.py
```

## 2. Server vorbereiten
Der Server hat eine REST-API und mehrere Endpunkte. Auf dem Server können mehrere Projekte angelegt. Dadurch können mehrere Umgebungen auf dem Server abgelegt werden.
Der Server hat Folgende Endpunkte:

| Pfad                              | Methode | Beschreibung                                                                                                                                                               | Beispiel-Body                                                                                                                                                                                                                         |
|-----------------------------------|---------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| /api/project                      | GET     | Listet alle Projekte auf                                                                                                                                                   |                                                                                                                                                                                                                                       |
| /api/project                      | POST    | Erstellt ein neues Projekt                                                                                                                                                 | ```{ "name": "Erdgeschoss Training", "description": "Lorem ipsum"}```                                                                                                                                                                 |
| /api/project/{project_id}         | PUT     | Aktualisiert ein  Projekt                                                                                                                                                  | ```{ "name": "Erdgeschoss Training", "description": "Lorem ipsum"}```                                                                                                                                                                 |
| /api/project/{project_id}         | DELETE  | Löscht ein  Projekt  und alle enthaltenen Messdaten                                                                                                                        |                                                                                                                                                                                                                                       |
| /api/wifi?project_id={project_id} | GET     | Gibt alle Messdaten zu dem angefragten Projekt zurück                                                                                                                      |                                                                                                                                                                                                                                       |
| /api/wifi                         | POST    | Speichert Messdaten im angegebenen Projekt ab. Die ```sensor_id``` ist ein Feld welches nicht genutzt wird. Es kann dafür verwendet werden mehrere Rover zu unterscheiden. | ```{"sensor_id":1,"timestamp":"2023-10-24T15:45:00","position_x":0,"position_y":0,"project_id":1,"signals":[{"bssid":"00:11:22:33:44:55","essid":"WLAN_A","rssi":-70},{"bssid":"66:77:88:99:AA:BB","essid":"WLAN_B","rssi":-80}]} ``` |
| /api/train                        | POST    | Trainiert einen RandomForestRegressor ohne weitere Parameteroptimierung und speichert das Modell für Predictions                                                           | ```{"project_id":1,"essids":["19BC","1209","1B45","1AD3"]}```                                                                                                                                                                         |
| /api/predict                      | POST    | Erstellt eine Prediction mit den übergebenen Messwerten                                                                                                                    | ```[{"00:11:22:33:44:55":-48,"66:77:88:99:AA:BB":-39}]```                                                                                                                                                                             |

### Projekt anlegen
Mit Folgendem Befehl kann ein neues Projekt angelegt werden:
```bash
curl -X POST http://localhost:3001/api/project \
-H "Content-Type: application/json" \
-d '{"name": "Erdgeschoss Training", "description": "Lorem ipsum"}'
```

## 2. Training - Daten aufnehmen

Ein RaspberryPi (im Folgenden "Rover") wurde ein USB-WLAN-Stick angeschlossen, um eine bessere Antenne zu haben. Auf den Rover haben wir die Skripte unter rover/ kopiert.
Mit dem Skript "train.py" können nun Strukturiert Daten erfasst werden. Zum Trainieren ist es wichtig die Koordinaten des Punktes an dem gemessen wird zu kennen. An jedem Punkt werden nun 5 aufeinanderfolgende Messungen durchgeführt. Zum starten einer Messung muss folgender Befehl ausgeführt werden:
```bash
sudo python3 train.py [sensor_id] [project_id] [x-Position] [y-Position]
```

Der Parameter ```sensor_id``` muss eine Ganzzahl (int) sein. Sie kann zum auseinanderhalten mehrere Rover verwendet werden, hat aber sonst keinen Einfluss auf Messungen oder eine spätere Auswertung.

Der Parameter ```project_id``` muss eine Ganzzahl (int) sein. Sie kann zum auseinanderhalten mehrere Projekte/Versuche verwendet werden, hat aber sonst keinen Einfluss auf Messung. Wir haben diesen Parameter für verschiedene Versuche verwendet, um die Daten auseinander zuhalten.

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from metrics import print_metrics

import warnings
warnings.filterwarnings('ignore')

Die Variable ```project_id``` enthält die Projekt-ID aus der die zum Training geladen werden sollen.

Die Variable ```essids``` wird zum filtern der Daten verwendet, damit ausschließlich die angegebenen ESSIDs verwendet werden.

In [2]:
project_id = 3
essids = ['eduroam']

### Daten laden
Aus der SQLite-Datenbank werden die Daten geladen und in einen Datensatz pro Messung umgewandelt.

In [3]:
engine = create_engine('sqlite:///sensordaten.db')
query = "SELECT sensor_id, timestamp, position_x, position_y, bssid, rssi, project_id, essid FROM wifi_signal_data WHERE project_id={} and essid in ({})".format(
    project_id, ', '.join(f"'{essid}'" for essid in essids))
df = pd.read_sql(query, engine)

In [4]:
pivot_df = df.pivot_table(
    index=['sensor_id', 'timestamp', 'position_x', 'position_y'],
    columns='bssid',
    values='rssi'
)
pivot_df = pivot_df.reset_index()

Die Datensätze werden in die Eingabefeatures X und Ausgabefeatures y getrennt

In [5]:
X = pivot_df.drop(['sensor_id', 'timestamp', 'position_x', 'position_y'], axis=1)
y = pivot_df[['position_x', 'position_y']].copy()
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 45 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   00:4E:35:2F:D9:80  92 non-null     float64
 1   24:62:CE:18:74:00  73 non-null     float64
 2   24:62:CE:18:9C:20  118 non-null    float64
 3   24:62:CE:18:D6:C0  385 non-null    float64
 4   24:62:CE:18:E1:40  7 non-null      float64
 5   24:62:CE:18:F1:80  238 non-null    float64
 6   24:62:CE:19:13:80  62 non-null     float64
 7   24:62:CE:19:B7:E1  235 non-null    float64
 8   24:62:CE:1A:20:40  12 non-null     float64
 9   24:62:CE:1A:64:80  353 non-null    float64
 10  24:62:CE:28:55:A0  16 non-null     float64
 11  24:62:CE:28:63:00  11 non-null     float64
 12  24:62:CE:28:98:20  36 non-null     float64
 13  24:62:CE:28:C2:01  292 non-null    float64
 14  24:62:CE:28:D5:60  20 non-null     float64
 15  24:62:CE:29:D5:A0  82 non-null     float64
 16  24:62:CE:29:D7:20  112 non

Die Daten werden in einen Trainings- und einen Testdatensatz zufällig geteilt. Die Testdaten sind 20% der Daten und werden nicht für das Trainieren des Modells verwendet.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

## Trainieren eines RandomForestRegressor
Als erstes wird ein RandomForestRegressor trainiert und mit den Testdaten werden Fehlerkennzahlen berechnet.

In [7]:
regr = RandomForestRegressor(random_state=0)
regr.fit(X_train, y_train)
y_pred = regr.predict(X_test)
print_metrics(y_pred, y_test)

RMSE: 3.854150154711674
RMSE X: 2.2123931712748974 RMSE Y: 1.6773839051092476
MAE X: 1.3696202531645576 MAE Y: 1.0909198312236286
Mean Position Error: 1.947209998409196
Median Position Error: 1.5579473675320359


## Test mit Scaling und meheren Modellen
Hier sind eine vielzahl von Modellen aufgeführt die mit verschiedenen Parametern getestet wurden.

In [8]:
pipeline = make_pipeline(MinMaxScaler(feature_range=(-90, -30)), RandomForestRegressor(n_estimators=100, min_samples_split=2, random_state=0))
# pipeline = make_pipeline(StandardScaler(), RandomForestRegressor(random_state=0))
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-90), MinMaxScaler(), KNeighborsRegressor())
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), StandardScaler(), KNeighborsRegressor())
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), StandardScaler(), MultiOutputRegressor(SVR(C=30, epsilon=0.3)))
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), MinMaxScaler(feature_range=(-70, -20)), MultiOutputRegressor(SVR(C=30, epsilon=0.5)))
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), MinMaxScaler(feature_range=(-70, -20)), MultiOutputRegressor(HistGradientBoostingRegressor(random_state=0)))
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), StandardScaler(), MultiOutputRegressor(HistGradientBoostingRegressor(random_state=0)))
# Optimized
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), StandardScaler(), MultiOutputRegressor(HistGradientBoostingRegressor(random_state=0, max_leaf_nodes=31, learning_rate=0.2)))
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), StandardScaler(), MLPRegressor(hidden_layer_sizes=(600,600,600), activation='relu', solver='adam', max_iter=500, random_state=0))
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-100), StandardScaler(), MLPRegressor(hidden_layer_sizes=(600,600,600), activation='relu', solver='adam', max_iter=500, random_state=0))
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), MinMaxScaler(feature_range=(-70, -20)), MultiOutputRegressor(Lasso() ))
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), MinMaxScaler(feature_range=(-80, -30)), MultiOutputRegressor(SVR(kernel='poly', C=10, epsilon=0.9, degree=10)))
# pipeline = make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), MinMaxScaler(feature_range=(-80, -30)), MultiOutputRegressor(SVR(C=30, epsilon=0.5)))

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print_metrics(y_pred, y_test)

RMSE: 3.8127316568213785
RMSE X: 2.19783092149821 RMSE Y: 1.67182611361023
MAE X: 1.374177215189874 MAE Y: 1.094210970464135
Mean Position Error: 1.9506508365643507
Median Position Error: 1.5579473675320359


# Kombination von mehreren Modellen

Nach vielem ausprobieren haben sich mehrere Modelle als ähnlich gut herausgestellt. Im Folgenden wird versucht verschiedene Modelle zu kombinieren, um ein besseres und robusteres Ergebnis zu erziehlen.

In [9]:
# Pipeline definieren
estimators = [
    ("rf", make_pipeline(MinMaxScaler(feature_range=(-70, -20)),
                         RandomForestRegressor(random_state=0, n_estimators=100, min_samples_split=2))),
    ("gb", make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95), StandardScaler(),
                         HistGradientBoostingRegressor(random_state=0, max_leaf_nodes=31, learning_rate=0.2))),
    ("svr", make_pipeline(SimpleImputer(missing_values=np.nan, strategy="constant", fill_value=-95),
                          MinMaxScaler(feature_range=(-70, -20)), SVR(kernel='poly', C=10, epsilon=0.9, degree=10))),
]

pipeline = MultiOutputRegressor(StackingRegressor(estimators=estimators, final_estimator=KNeighborsRegressor(n_jobs=-1, n_neighbors=3)))

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print_metrics(y_pred, y_test)

RMSE: 1.7663009845288329
RMSE X: 1.5075991289512698 RMSE Y: 1.1223844419106306
MAE X: 0.9957805907172999 MAE Y: 0.5738396624472574
Mean Position Error: 1.3325402228102854
Median Position Error: 1.3333333333333335


## Test mit weiteren Daten

In einem weiteren Projekt wurden zusätzlich weitere Daten erfasst. Es können nun einzelne Punkte geladen werden und mit dem Trainierten Modell vorhergesagt werden.

In [10]:
df_test = pd.DataFrame(columns=X_train.columns)

project_id = 4
pos_x = "46"
pos_y = "2"
query_test = "SELECT sensor_id, timestamp, position_x, position_y, bssid, rssi, project_id, essid FROM wifi_signal_data WHERE project_id={} and essid in ({}) and position_x = {} and position_y = {}".format(
    project_id, ', '.join(f"'{essid}'" for essid in essids), pos_x, pos_y)
df_test_db = pd.read_sql(query_test, engine)

for i, row in df_test_db.iterrows():
    if row['bssid'] in df_test.columns:
        df_test.loc[0, row['bssid']] = row['rssi']

pipeline.predict(df_test)

array([[46.66666667, -0.93333333]])